# 🧠 E-Prop Spiking Neural Network

E-Prop (Eligibility Propagation) enables online learning in SNNs without BPTT:

$$\Delta w = \eta \cdot L_j \cdot e_{ji}$$
$$\frac{de_{ji}}{dt} = -\frac{e_{ji}}{\tau_e} + \psi_j(t) \cdot z_i(t)$$

Where:
- $L_j$ = learning signal (error projected through random feedback)
- $e_{ji}$ = eligibility trace (low-pass filtered pre-post correlation)
- $\psi_j(t)$ = surrogate gradient $\sigma'(v_j)$

Based on: Bellec et al. (2020), *Nature Communications*.


## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
try:
    import taas_checkmate as ne
except ImportError:
    import subprocess
    subprocess.run(["maturin", "develop", "--release", "-q"], check=True)
    import taas_checkmate as ne

plt.style.use("dark_background")
print("✅ Setup complete")


## Create E-Prop SNN

In [ ]:
N_INPUT = 2
N_HIDDEN = 32
N_PAD = (N_HIDDEN + 7) // 8 * 8
DT = 1.0

# E-Prop layer
layer = ne.PyEPropLIF(N_PAD, dt=DT, tau_m=10.0, tau_e=20.0)
print(f"E-Prop LIF: {N_PAD} hidden neurons (SIMD-padded)")
print(f"  tau_m={10.0}, tau_e={20.0}")

# Weights
rng = np.random.default_rng(42)
w_in = rng.uniform(-1.0, 1.0, (N_PAD, N_INPUT)).astype(np.float32) * 0.5
w_out = np.zeros(N_PAD, dtype=np.float32)  # readout
w_fb = rng.uniform(-0.3, 0.3, N_PAD).astype(np.float32)  # feedback

# XOR dataset
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=np.float32)
y = np.array([0, 1, 1, 0], dtype=np.float32)

# Buffers
I = np.zeros(N_PAD, dtype=np.float32)
spikes = np.zeros(N_PAD, dtype=np.int32)
elig = np.zeros(N_PAD, dtype=np.float32)
ls = np.zeros(N_PAD, dtype=np.float32)  # learning signals

print(f"\nXOR dataset: {len(X)} patterns")


## Training Loop

In [ ]:
N_EPOCHS = 100
LR = 0.02

errors = []
accuracy_history = []

from rich.progress import Progress

with Progress() as pbar:
    task = pbar.add_task("[cyan]Training...", total=N_EPOCHS)

    for epoch in range(N_EPOCHS):
        epoch_errors = []
        correct = 0

        for i in range(4):
            layer.reset()
            pred = 0.0

            for _ in range(5):  # rate coding
                # Forward
                I[:N_INPUT] = w_in[:, :N_INPUT] @ X[i]
                layer.step(I, spikes, np.zeros(N_PAD, dtype=np.int32))

                # Readout
                pred += float(spikes[:N_HIDDEN] @ w_out[:N_HIDDEN])

            pred /= 5
            error = y[i] - pred

            # Learning signal
            ls[:] = error * w_fb

            # Update readout
            w_out[:N_HIDDEN] += LR * error * spikes[:N_HIDDEN].astype(np.float32)

            # E-Prop weight update
            layer.copy_eligibility(elig)
            # Δw_in = LR * LS[j] * elig[j] * input[i]
            for j in range(N_PAD):
                w_in[j, :N_INPUT] += LR * ls[j] * elig[j] * X[i]

            epoch_errors.append(abs(error))
            if abs(error) < 0.5:
                correct += 1

        avg_err = np.mean(epoch_errors)
        errors.append(avg_err)
        accuracy_history.append(correct / 4)
        pbar.update(task, advance=1)

print(f"\nTraining complete!")
print(f"  Final error: {errors[-1]:.6f}")
print(f"  Final accuracy: {accuracy_history[-1]*100:.0f}%")


## Learning Curve

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.plot(errors, color="#00ff88", linewidth=1)
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Mean Absolute Error")
ax1.set_title("E-Prop Learning Curve")
ax1.grid(alpha=0.1)

ax2.plot(np.arange(len(accuracy_history)) * 10,
         np.array(accuracy_history) * 100,
         color="#ff6600", linewidth=1)
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy (%)")
ax2.set_title("XOR Classification Accuracy")
ax2.set_ylim(0, 105)
ax2.axhline(100, color="green", linestyle="--", alpha=0.5, label="100%")
ax2.axhline(50, color="red", linestyle="--", alpha=0.3, label="Chance")
ax2.legend()
ax2.grid(alpha=0.1)

plt.tight_layout()
plt.show()


## Test XOR

In [ ]:
print("═" * 40)
print("XOR Test Results")
print("═" * 40)

for i in range(4):
    layer.reset()
    preds = []
    for _ in range(10):
        I[:N_INPUT] = w_in[:, :N_INPUT] @ X[i]
        layer.step(I, spikes, np.zeros(N_PAD, dtype=np.int32))
        preds.append(float(spikes[:N_HIDDEN] @ w_out[:N_HIDDEN]))

    avg_pred = np.mean(preds)
    print(f"  {X[i][0]} XOR {X[i][1]} = {avg_pred:.4f}  (target={y[i]:.0f})  {'✅' if abs(avg_pred - y[i]) < 0.5 else '❌'}")

print("═" * 40)


## Eligibility Traces Analysis

In [ ]:
# Visualize eligibility traces
layer.reset()
elig_history = np.zeros((100, N_PAD), dtype=np.float32)

for t in range(100):
    inp = X[t % 4]
    I[:N_INPUT] = w_in[:, :N_INPUT] @ inp
    layer.step(I, spikes, spikes.copy())
    layer.copy_eligibility(elig)
    elig_history[t] = elig

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 6))

im = ax1.imshow(elig_history[:, :16].T, aspect="auto", cmap="RdBu_r")
ax1.set_xlabel("Time Step")
ax1.set_ylabel("Neuron")
ax1.set_title("Eligibility Traces (first 16 neurons)")
plt.colorbar(im, ax=ax1, shrink=0.8)

# Mean eligibility over time
ax2.plot(np.mean(elig_history, axis=1), color="#00ff88", linewidth=0.8)
ax2.set_xlabel("Time Step")
ax2.set_ylabel("Mean Eligibility")
ax2.set_title("Mean Eligibility Trace Dynamics")
ax2.grid(alpha=0.1)

plt.tight_layout()
plt.show()
